# Fixed-wing drone — wing vibration, cyclic loads and life over three mission types

Notebook 09 sized the wing for a pull-up and an engine-out case; notebook 12 gave the propellers'
rpm and unbalance. This notebook asks how long the wing lasts, and finds something on the way: one
wing mode sits on the cruise shaft frequency. The chain is the same as in notebook 13:

```
propeller data (12) ─► 1P / 2P lines, unbalance force per nacelle
wing + nacelle masses ─► modal analysis (Talos) ─► Campbell diagram, margins ─► a design decision
three missions (Chronos) ─► rainflow + amplified vibration cycles ─► load spectrum per mission
unit FEA cases (Talos) ─► damage per mission, hotspot ─► static re-check ─► fleet life
```

All inputs are explicit assumptions (lumped masses, 3 % damping, an S-N curve for LW-PLA, load
levels in g). Compare, record, then test.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz

RUNS = Path("_runs/fixed_wing_life"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "fixed_wing.py"
shutil.copy(Path("designs/fixed_wing.py"), design_file)
drone = dedalus.load_design(f"{design_file}:FixedWing")

# ---- hand-copied inputs ------------------------------------------------------------------------
PREFERRED = dict(thickness=0.15)                              # notebook 09: NACA 2415 preferred
AUW_KG, WING_AREA_M2 = 1.35, 0.17                             # notebook 09
PROP = {                                                      # notebook 12: _runs/propeller/fw_9x6.json, points
    "cruise":     {"rpm": 5959.0, "thrust_N": 0.89, "unbalance_N": 0.18},
    "climb":      {"rpm": 8941.0, "thrust_N": 5.09, "unbalance_N": 0.27},
    "engine_out": {"rpm": 8998.0, "thrust_N": 4.57, "unbalance_N": 0.27},
}
BLADES = 2
NACELLE_MASS_T = 70e-6                                        # motor + prop + ESC per nacelle, tonnes
W = AUW_KG * 9.81
LIFT_UNIT_MPA = W / (WING_AREA_M2 * 1e6)                      # pressure for n = 1 (level flight) in MPa

## 1. Wing, masses, one mesh

In [ ]:
p = drone.resolve(part="wing", **PREFERRED)
wing = drone.generate(part="wing", **PREFERRED)
cad = wing.export(RUNS / "cad")
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed; replace with coupon tests")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

yc, ny, d, f = p["fuselage_diameter"] / 2 + 5.0, p["nacelle_y"], p["nacelle_diameter"] / 2 + 1, p["nacelle_forward"]
REGIONS = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)),
           talos.Surfaces("lift", lower_skins(cad.artifacts["step"])),
           talos.SurfacesInBox("motor_left", (-f - 0.1, -ny - d, -d, -f + 0.1, -ny + d, d)),
           talos.SurfacesInBox("motor_right", (-f - 0.1, ny - d, -d, -f + 0.1, ny + d, d))]
MASSES = [talos.PointMass("motor_left", NACELLE_MASS_T), talos.PointMass("motor_right", NACELLE_MASS_T)]
MESH = talos.MeshSettings(element_size=12.0)

def model(loads, name, step=None):
    return talos.StructuralModel(step or cad.artifacts["step"], "mm-N-MPa", LW_PLA, REGIONS, [talos.FixedSupport("root")],
                                 loads, MESH, name=name, masses=MASSES)

base = model([], "modal")
print(base.mesh(RUNS / "mesh", progress=True))

def case_dir(name):
    d_ = RUNS / name
    if not d_.exists():
        shutil.copytree(RUNS / "mesh", d_)
    return d_

## 2. Modes with the nacelles on the wing, and the Campbell diagram

In [ ]:
modes = base.solve_modes(case_dir("modal"), n_modes=8, progress=True)
freqs = modes.metrics["frequencies_hz"]
print([round(x, 1) for x in freqs])
tviz.show(tviz.plot_mode(modes, mode=1))     # first wing bending

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=7))     # the mode near the cruise shaft frequency

In [ ]:
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03, source="Talos modal, LW-PLA solid-equivalent E, nacelle masses")
fig = structure.campbell({"1P": 1, "2P": BLADES}, np.linspace(2000, 10000, 30), operating_rpm={k: v["rpm"] for k, v in PROP.items()})
lines = {k: v["rpm"] / 60 for k, v in PROP.items()}
margins = pd.DataFrame({k: {"1P_hz": fq, "nearest_mode_hz": structure.nearest_mode(fq), "margin": structure.margin(fq),
                            "amplification": float(structure.amplification(fq)[0])} for k, fq in lines.items()}).round(2)
margins

### A finding, and a decision

The cruise shaft frequency (≈ 99 Hz) falls within a few percent of a wing mode: the dynamic
amplification is an order of magnitude, the usual comfort margin is 20 %. Three ways out, all
engineer decisions, none of them a change the software makes on its own:

1. **move the excitation** — cruise at a different rpm (a different propeller pitch, notebook 12);
2. **move the mode** — a stiffer or lighter nacelle mount, a spar;
3. **accept and verify** — run the fatigue with the amplification and see whether the life is still acceptable.

This notebook does (3) with the amplified vibration in the spectrum, and at the end repeats the
assessment for the thinner NACA 2412 wing to show how far a design change moves the numbers.

## 3. Unit load cases

| pattern | unit case | level unit |
|---|---|---|
| `lift` | pressure for n = 1 on the outer-panel lower skins | load factor n |
| `thrust` | +6 N on both nacelle noses | N per motor |
| `thrust_left` | +8 N on the left nacelle only (engine-out) | N |
| `vib_left` | 1 N vertical at the left nacelle nose (rotor unbalance) | N |

In [ ]:
UNIT = {
    "lift":        (model([talos.Pressure("lift", LIFT_UNIT_MPA)], "lift"), 1.0),
    "thrust":      (model([talos.Force("motor_left", fx=6.0), talos.Force("motor_right", fx=6.0)], "thrust"), 6.0),
    "thrust_left": (model([talos.Force("motor_left", fx=8.0)], "thrust_left"), 8.0),
    "vib_left":    (model([talos.Force("motor_left", fz=1.0)], "vib_left"), 1.0),
}
unit_results = {}
for name, (m, load) in tqdm(UNIT.items(), desc="unit cases"):
    unit_results[name] = m.solve(case_dir(name))
    r = unit_results[name]
    print(f"{name:<12} {'ok' if r.ok else 'FAILED'}  max von Mises {r.metrics.get('max_von_mises', float('nan')):.3f} MPa at level {load:g}")
unit_cases = {k: (unit_results[k], UNIT[k][1]) for k in UNIT}
tviz.show(tviz.plot_results(unit_results["vib_left"], field="von_mises"))

## 4. Three missions

`lift` levels are load factors (1 = level flight, 1.4 = a 45° banked turn, 2.5 = the pull-up of
notebook 09); gusts are `repeat`ed excursions. Vibration: unbalance on the left nacelle at the shaft
frequency of the segment's rpm (right nacelle assumed identical by symmetry — the spectrum counts one
side; the hotspot map shows where).

In [ ]:
def unb(point):
    return chronos.Excitation(f"unbalance {point}", PROP[point]["rpm"] / 60, PROP[point]["unbalance_N"], "vib_left")

TC, TCL = PROP["cruise"]["thrust_N"], PROP["climb"]["thrust_N"]
missions = {
    "survey": chronos.Mission("survey", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("mapping legs", 2400, {"lift": 1.0, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("turn between legs", 8, {"lift": 1.3, "thrust": TC}, (unb("cruise"),), repeat=16),
        chronos.Segment("light turbulence", 2, {"lift": 1.25, "thrust": TC}, (unb("cruise"),), repeat=80),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 1.8}),
    ), "45 min mapping survey: long straight legs, gentle turns"),
    "patrol": chronos.Mission("patrol", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("loiter", 1500, {"lift": 1.15, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("steep turn", 6, {"lift": 1.6, "thrust": TC}, (unb("cruise"),), repeat=60),
        chronos.Segment("evasive pull-up", 2, {"lift": 2.5, "thrust": TCL}, (unb("climb"),), repeat=4),
        chronos.Segment("engine-out drill", 30, {"lift": 1.0, "thrust_left": PROP["engine_out"]["thrust_N"]}, (unb("engine_out"),)),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 2.0}),
    ), "30 min patrol: continuous loiter turns, a few hard pull-ups, one engine-out drill"),
    "windy_hops": chronos.Mission("windy_hops", (
        chronos.Segment("take-off + climb", 45, {"lift": 1.3, "thrust": TCL}, (unb("climb"),), repeat=6),
        chronos.Segment("short cruise", 300, {"lift": 1.0, "thrust": TC}, (unb("cruise"),), repeat=6),
        chronos.Segment("gust", 1.5, {"lift": 1.7, "thrust": TC}, (unb("cruise"),), repeat=300),
        chronos.Segment("hard touchdown", 1, {"lift": 2.5}, repeat=6),
    ), "six short hops in gusty wind: 300 gusts, six hard touchdowns, ~35 min"),
}
for m in missions.values():
    fig = m.profile(patterns=["lift", "thrust", "thrust_left"])
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "segments": len(m.segments)} for k, m in missions.items()}).T

In [ ]:
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k}.json")
    fig = sp.plot()
spectra["patrol"].table().round(4)

## 5. Damage per mission, hotspot, static re-check

S-N for LW-PLA — **assumed**: σ_f = 22 MPa, b = −0.12, Goodman with 14 MPa ultimate. Printed foamed
PLA is weak in fatigue and sensitive to temperature; coupon tests first.

In [ ]:
CURVE = talos.FatigueCurve("LW-PLA (assumed)", sigma_f=22.0, b=-0.12, ultimate=14.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, spectra[k].to_dict(), CURVE, workdir=RUNS / f"fatigue_{k}") for k in tqdm(missions, desc="fatigue")}
life = pd.DataFrame({k: {"duration_min": missions[k].duration_h * 60, "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"],
                         "hotspot": tuple(round(x, 1) for x in f.result.metrics["hotspot_location"])} for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_results["lift"].artifacts["mesh"]))
contrib = pd.Series(fatigue[worst].contributions).sort_values(ascending=False)
ax = contrib.head(8).plot.barh(figsize=(8, 3.2), title=f"{worst}: damage contributions at the hotspot"); ax.invert_yaxis()

In [ ]:
hot = fatigue[worst].hotspot
unit_vm = {k: float(talos.read_frd(r.artifacts["frd"]).von_mises[hot]) / UNIT[k][1] for k, r in unit_results.items()}
rows = {}
for k, sp in spectra.items():
    peak = {}
    for b in sp.blocks:
        peak[b.pattern] = max(peak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
    stress = sum(peak.get(pat, 0.0) * unit_vm[pat] for pat in unit_vm)
    rows[k] = {**{f"peak_{pat}": peak.get(pat, 0.0) for pat in unit_vm}, "hotspot_stress_MPa": stress, "SF_yield": LW_PLA.yield_strength / stress}
pd.DataFrame(rows).T.round(2)

## 6. What the numbers say — and the vibration amplitude the resonance really causes

Read the damage column first: at ~10⁻¹⁵ per mission this wing is **not fatigue-limited** — the
solid-equivalent stresses are a fraction of a MPa against an S-N knee of tens of MPa. That is a
result, not a failure of the method: the spectrum, the hotspot and the ranking of the missions are
recorded, and the same pipeline will bite when the wing becomes a 1.2 mm printed shell (the stresses
scale with the section modulus ratio, an input for the next revision).

What the resonance *does* cause is motion: the nacelle oscillates at the shaft frequency with an
amplitude = amplification × unbalance force × static compliance. That is the number to compare with
what the camera and the propeller bearings tolerate. The table gives it per operating point for the
wing as designed, with the resonance avoided (cruise rpm moved), and for the thinner NACA 2412 wing.

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: m.duration_h for k, m in missions.items()}
usage = {"survey": 0.5, "patrol": 0.3, "windy_hops": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
print(f"damage per 1000 flight hours with usage {usage}: {rate * 1000:.3g}  ->  "
      f"{'not life-limiting (> 1e6 h)' if rate * 1e6 < 1 else f'{1 / rate:.0f} h to failure'}")
sim = chronos.simulate_life(damage, hours, usage, n_flights=2000, seed=0)
fig = sim.plot()

In [ ]:
def assess_design(step, label):
    # a new mesh, modal analysis and unit cases for another wing STEP; same regions, masses, missions, curve
    work = RUNS / label
    m0 = model([], "modal", step); m0.mesh(work / "mesh")
    md_ = m0.solve_modes(work / "mesh", n_modes=8)
    st = chronos.Structure(tuple(md_.metrics["frequencies_hz"]), 0.03)
    units = {}
    for name, (mm, load) in UNIT.items():
        shutil.copytree(work / "mesh", work / name)
        units[name] = (model(mm.loads, name, step).solve(work / name), load)
    dmg = {kk: talos.assess_fatigue(units, chronos.build_spectrum(mission, st).to_dict(), CURVE).result.metrics["damage_per_pass"]
           for kk, mission in missions.items()}
    return st, units, dmg

no_resonance = {kk: talos.assess_fatigue(unit_cases, chronos.build_spectrum(m, None).to_dict(), CURVE).result.metrics["damage_per_pass"]
                for kk, m in missions.items()}
thin_step = drone.generate(part="wing", thickness=0.12).export_step(RUNS / "cad" / "wing_2412.step")
thin_structure, thin_units, thin_damage = assess_design(thin_step, "naca2412")
print("NACA 2412 modes:", [round(x, 1) for x in thin_structure.modes_hz])

def nacelle_amplitude_mm(st, units, point):
    compliance = units["vib_left"][0].metrics["max_displacement"] / UNIT["vib_left"][1]        # mm per N, static
    f = PROP[point]["rpm"] / 60
    daf = float(st.amplification(f)[0]) if st is not None else 1.0
    return daf * PROP[point]["unbalance_N"] * compliance

variants = {"NACA 2415 (as is)": (structure, unit_cases, damage), "NACA 2415, cruise rpm moved off the mode": (None, unit_cases, no_resonance),
            "NACA 2412 (thinner)": (thin_structure, thin_units, thin_damage)}
rows = {}
for name, (st, units, dmg) in variants.items():
    r = {f"nacelle amplitude {pt} [mm]": nacelle_amplitude_mm(st, units, pt) for pt in PROP}
    r["damage per 1000 h (usage mix)"] = sum(usage[kk] * dmg[kk] for kk in usage) / sum(usage[kk] * hours[kk] for kk in usage) * 1000
    rows[name] = r
pd.DataFrame(rows).T

## 7. Export

In [ ]:
summary = {"design": {"file": "designs/fixed_wing.py", "parameters": p}, "modes_hz": freqs, "damping_ratio": 0.03,
           "excitations_hz": lines, "margins": margins.to_dict(),
           "unit_cases": {k: {"load": UNIT[k][1], "max_von_mises": unit_results[k].metrics["max_von_mises"]} for k in UNIT},
           "curve": CURVE.__dict__, "missions": {k: m.describe() for k, m in missions.items()},
           "damage_per_mission": damage, "damage_no_resonance": no_resonance, "damage_naca2412": thin_damage,
           "hours_per_mission": hours, "usage": usage, "damage_per_1000h": rate * 1000, "nacelle_amplitude_mm": rows,
           "spectra": {k: str(RUNS / f"spectrum_{k}.json") for k in missions}}
(RUNS / "life.json").write_text(json.dumps(summary, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Reading the result:** the amplitude column is the design lever — a mode on the cruise shaft
frequency turns a 0.2 N unbalance into millimetres of nacelle motion, and moving the rpm (or the mode)
removes it; the damage column says fatigue is not what limits this wing. Both are recorded with the
missions and curves that produced them. The decision, and the revision that follows, is yours — this
notebook only makes it visible and repeatable.